# ARC Prize 2026 (ARC-AGI-2) — Notebook de entrega

Corre **offline** (sin internet). Pipeline: cargar test → lanzar **1 worker por GPU**
(sharding de tareas, sin DDP) → por tarea: **TTT** + inferencia multi-candidato con
**scoring por log-verosimilitud augmentada** → merge de shards → `submission.json`.

Robustez: cada worker pre-escribe la baseline de su shard y reescribe el JSON tras cada
tarea (escritura incremental), con fallback granular por tarea y presupuesto de tiempo
con degradación (menos augmentaciones/candidatos/steps de TTT si el reloj aprieta).

**Antes de enviar**, adjunta como *Datasets* de este notebook:
1. La competencia `arc-prize-2026-arc-agi-2` (datos).
2. El código de este repo (`src/` + `scripts/` como dataset, p.ej. `arc-agi2-code`).
3. Los pesos del modelo base + LoRA entrenado (p.ej. `arc-agi2-model`).

Si el modelo no está disponible, cae a la baseline y **siempre** escribe un submission válido.

In [ ]:
# === Configuración ===
import os, sys, glob, json, time, subprocess
from pathlib import Path

# Datos de la competencia
COMP_DIR = '/kaggle/input/arc-prize-2026-arc-agi-2'
if not os.path.isdir(COMP_DIR):
    COMP_DIR = str(Path.cwd().parent / 'data')  # fallback local
TEST_CHALLENGES = os.path.join(COMP_DIR, 'arc-agi_test_challenges.json')

# Código del repo (dataset con src/arc y scripts/ como hermanos)
CODE_DIR = None
for c in glob.glob('/kaggle/input/*/src') + ['../src', 'src']:
    if os.path.isdir(os.path.join(c, 'arc')):
        CODE_DIR = os.path.abspath(c); break
assert CODE_DIR, 'No se encontró src/arc. Adjunta el código como dataset.'
sys.path.insert(0, CODE_DIR)
SCRIPTS_DIR = os.path.join(os.path.dirname(CODE_DIR), 'scripts')
WORKER = os.path.join(SCRIPTS_DIR, 'infer_worker.py')
assert os.path.isfile(WORKER), f'No se encontró {WORKER}'

# Modelo base (config.json + pesos, sin adapter) y adaptador LoRA
MODEL_DIR = None
for cfg in glob.glob('/kaggle/input/**/config.json', recursive=True):
    d = os.path.dirname(cfg); fs = os.listdir(d)
    if any(f.endswith(('.safetensors', '.bin')) for f in fs) and 'adapter_config.json' not in fs:
        MODEL_DIR = d; break
ad = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
LORA_DIR = os.path.dirname(ad[0]) if ad else None

# --- Flags principales ---
ENABLE_TTT = True      # TTT por tarea: la técnica común a los 3 ganadores 2025
SELECTION = 'score'    # 'score' = multi-candidato + ranking por log-verosimilitud
N_AUG = 4              # vistas augmentadas de generación
N_CANDS = 2            # candidatos por vista (1 greedy + sampling)
SCORE_VIEWS = 4        # vistas para el scoring de candidatos
TTT_STEPS = 16
MAX_NEW_TOKENS = 1024
# Presupuesto de pared de los workers. La fase "Save & Run All" consume quota GPU
# propia (el rerun de la submission NO): mantenerlo acotado; en L4x4 quota corre al 2x.
TIME_BUDGET_S = int(1.0 * 3600)
OUT_PATH = 'submission.json'

try:
    import torch
    N_GPUS = torch.cuda.device_count()
except Exception:
    N_GPUS = 0
print('COMP_DIR =', COMP_DIR); print('CODE_DIR =', CODE_DIR)
print('MODEL_DIR =', MODEL_DIR); print('LORA_DIR =', LORA_DIR)
print('N_GPUS =', N_GPUS)

In [ ]:
# === Cargar tareas de test ===
from arc.data import load_dataset
tasks = load_dataset(TEST_CHALLENGES)
print('Tareas de test:', len(tasks))

In [ ]:
# === Inferencia: 1 worker por GPU (sharding de tareas, sin DDP) ===
USE_MODEL = MODEL_DIR is not None and N_GPUS >= 1
N_WORKERS = max(1, N_GPUS) if USE_MODEL else 0
shard_paths = [f'submission_shard_{i}.json' for i in range(max(1, N_WORKERS))]

if USE_MODEL:
    procs, logs = [], []
    for i in range(N_WORKERS):
        cmd = [sys.executable, WORKER,
               '--challenges', TEST_CHALLENGES, '--out', shard_paths[i],
               '--n-shards', str(N_WORKERS), '--shard-idx', str(i),
               '--model-dir', MODEL_DIR,
               '--selection', SELECTION, '--n-aug', str(N_AUG),
               '--n-cands', str(N_CANDS), '--score-views', str(SCORE_VIEWS),
               '--max-new-tokens', str(MAX_NEW_TOKENS),
               '--time-budget-s', str(TIME_BUDGET_S)]
        if LORA_DIR:
            cmd += ['--lora-dir', LORA_DIR]
        if ENABLE_TTT:
            cmd += ['--ttt', '--ttt-steps', str(TTT_STEPS)]
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), PYTHONUNBUFFERED='1')
        lf = open(f'worker_{i}.log', 'w')
        procs.append(subprocess.Popen(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT))
        logs.append(lf)
    print(f'{N_WORKERS} workers lanzados; presupuesto {TIME_BUDGET_S/3600:.1f} h')

    deadline = time.time() + TIME_BUDGET_S + 900  # margen para el flush final
    while any(p.poll() is None for p in procs):
        if time.time() > deadline:
            print('Deadline: terminando workers (los shards son incrementales).')
            for p in procs:
                if p.poll() is None:
                    p.terminate()
            break
        time.sleep(60)
        alive = sum(p.poll() is None for p in procs)
        print(f'[{time.strftime("%H:%M:%S")}] workers vivos: {alive}/{N_WORKERS}', flush=True)
    for p in procs:
        p.wait(timeout=120)
    for lf in logs:
        lf.close()
    for i in range(N_WORKERS):
        tail = Path(f'worker_{i}.log').read_text(encoding='utf-8').splitlines()[-5:]
        print(f'--- worker {i} (últimas líneas) ---')
        print('\n'.join(tail))
else:
    # Sin modelo/GPU: baseline para todas las tareas (run_shard con model=None)
    from arc.runner import run_shard
    run_shard(tasks, list(tasks), model=None, out_path=shard_paths[0])
    print('Sin modelo: baseline escrita en', shard_paths[0])

In [ ]:
# === Merge de shards + validación de formato ===
from arc.runner import merge_submissions
submission = merge_submissions(shard_paths, tasks)
with open(OUT_PATH, 'w') as f:
    json.dump(submission, f)

sub = json.load(open(OUT_PATH))
assert all(isinstance(v, list) for v in sub.values())
for tid, task in tasks.items():
    assert len(sub[tid]) == len(task.test), tid
    for att in sub[tid]:
        assert 'attempt_1' in att and 'attempt_2' in att
print('Formato OK:', len(sub), 'tareas ->', OUT_PATH)